OCENA BASELINE, AUGMENTACJA, HIPERPARAMETRY

In [ ]:
!nvidia-smi -L
%pip install -q "ultralytics>=8.3,<9" keras-tuner "codecarbon>=3,<4" "pandas>=2,<4"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
![ -d /content/garbage-yolo ] || git clone -q --depth 1 https://github.com/PovMiki/garbage-yolo /content/garbage-yolo

Dataset z repo a wyniki w MyDrive/garbage_yolo

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import time, yaml
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import keras_tuner as kt
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops
from codecarbon import OfflineEmissionsTracker

DS = Path("/content/garbage-yolo/dataset_clean")
OUT = Path("/content/drive/MyDrive/garbage_yolo")
RUNS = OUT / "runs"
RESULTS = OUT / "results_comparison.csv"
PER_CLASS = OUT / "results_per_class.csv"
BASELINE = OUT / "baseline_v0_best.pt"
EPOCHS, BATCH, IMGSZ, SEED = 50, 8, 640, 42
WORKERS, AMP, COUNTRY = 2, True, "POL"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
assert DEVICE == 0, "Brak GPU"
assert BASELINE.is_file(), f"Brak {BASELINE}"

cfg = yaml.safe_load((DS / "data.yaml").read_text())
cfg["path"] = str(DS)
DATA = DS / "data_colab.yaml"
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
TEST_IMAGES = sorted((DS / "test/images").glob("*"))
RUNS.mkdir(parents=True, exist_ok=True)

Ocena modelu
cpu_latency mierzy czas jednego zdj na CPU na 100 zdj testowych
Emmisions sumuje czas treningu kwh i CO2
Evaluate robi ocene na zbiorze
add_result dopisuje wynik

In [ ]:
def cpu_latency(model, images, repeats=3):
    model.predict(str(images[0]), imgsz=IMGSZ, device="cpu", verbose=False)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for img in images:
            model.predict(str(img), imgsz=IMGSZ, device="cpu", verbose=False)
        times.append((time.perf_counter() - start) / len(images) * 1000)
    return sum(times) / len(times), pd.Series(times).std()


def emissions(run):
    file = run / "emissions.csv"
    if not file.is_file():
        return {"train_min": None, "train_kWh": None, "train_CO2_kg": None}
    e = pd.read_csv(file)
    return {"train_min": e["duration"].sum() / 60, "train_kWh": e["energy_consumed"].sum(),
            "train_CO2_kg": e["emissions"].sum()}


def evaluate(weights, name, run):
    model = YOLO(str(weights))
    m = model.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
                  project=str(run), name="test", exist_ok=True, plots=True, verbose=False)
    p, r = float(m.box.mp), float(m.box.mr)
    ms, ms_std = cpu_latency(model, TEST_IMAGES[:100])
    row = {"model": name, "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
           "precision": p, "recall": r, "F1": 2 * p * r / (p + r + 1e-9),
           "ms_cpu": ms, "ms_cpu_std": ms_std, "FPS": 1000 / ms,
           "size_MB": os.path.getsize(weights) / 1e6,
           "GFLOPs": get_flops(model.model, IMGSZ),
           "params_M": sum(x.numel() for x in model.model.parameters()) / 1e6,
           **emissions(run)}
    per_class = pd.DataFrame({"model": name, "class": [m.names[int(c)] for c in m.box.ap_class_index],
                              "mAP50-95": [float(m.box.maps[int(c)]) for c in m.box.ap_class_index]})
    return row, per_class


def add_result(weights, name, run):
    df = pd.read_csv(RESULTS) if RESULTS.is_file() else pd.DataFrame(columns=["model"])
    if name in df["model"].values:
        return
    row, per_class = evaluate(weights, name, run)
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS, index=False)
    per_class.to_csv(PER_CLASS, mode="a", header=not PER_CLASS.is_file(), index=False)


add_result(BASELINE, "baseline", OUT)
pd.read_csv(RESULTS).round(4)

Augmentacja
Train trenuje YOLO z ustawieniami baseline, bez obrotow
Testowanie 3 wariantow jasnosc (0,4 0,7) i skala (0,5 0,8)

In [ ]:
def train(name, epochs=EPOCHS, **params):
    run = RUNS / name
    if (run / "done").exists():
        return run
    run.mkdir(parents=True, exist_ok=True)
    tracker = OfflineEmissionsTracker(project_name=name, country_iso_code=COUNTRY, output_dir=str(run),
                                      output_file="wyniki/emissions.csv", measure_power_secs=15, tracking_mode="machine",
                                      save_to_file=True, log_level="error")
    tracker.start()
    try:
        if (run / "weights/last.pt").exists():
            YOLO(str(run / "weights/last.pt")).train(resume=True)
        else:
            settings = dict(optimizer="AdamW", lr0=0.001, cos_lr=True, patience=10, degrees=0)
            settings.update(params)
            YOLO("yolo11n.pt").train(data=str(DATA), epochs=epochs, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                     workers=WORKERS, seed=SEED, deterministic=True, amp=AMP, cache=False,
                                     project=str(RUNS), name=name, exist_ok=True, plots=True, **settings)
    finally:
        tracker.stop()
    (run / "done").touch()
    return run


experiments = {
    "aug_brightness": {"hsv_v": 0.7},
    "aug_scale": {"scale": 0.8},
    "aug_both": {"hsv_v": 0.7, "scale": 0.8},
}

for name, params in experiments.items():
    run = train(name, **params)
    add_result(run / "weights/best.pt", name, run)

df = pd.read_csv(RESULTS).set_index("model")
change = ((df - df.loc["baseline"]) / df.loc["baseline"] * 100).round(1)
change[["mAP50", "mAP50-95", "precision", "recall", "F1"]]

Keras Tuner
Trenowanie przez 15 epok z randomowymi ustawieniami
Stojenie lr0, optymalizxator, weight_decay, jasnosc i skala
Tabela z najlepszymi ustawieniami

In [ ]:
class YoloTuner(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        params = {
            "lr0": hp.Choice("lr0", [0.01, 0.001, 0.0001]),
            "optimizer": hp.Choice("optimizer", ["AdamW", "SGD"]),
            "weight_decay": hp.Choice("weight_decay", [0.0005, 0.001, 0.005]),
            "hsv_v": hp.Float("hsv_v", 0.3, 0.8, step=0.1),
            "scale": hp.Float("scale", 0.3, 0.9, step=0.2),
        }
        run = train(f"tuner_{trial.trial_id}", epochs=15, **params)
        m = YOLO(str(run / "weights/best.pt")).val(data=str(DATA), split="val", imgsz=IMGSZ, batch=BATCH,
                                                   device=DEVICE, workers=WORKERS, plots=False, verbose=False)
        return {"val_map": float(m.box.map)}


tuner = YoloTuner(objective=kt.Objective("val_map", direction="max"), max_trials=5, seed=SEED,
                  directory=str(OUT / "my_dir"), project_name="yolo_tuning")
tuner.search()

trials = tuner.oracle.get_best_trials(num_trials=5)
pd.DataFrame([{"trial": t.trial_id, **t.hyperparameters.values, "val_mAP50-95": t.score} for t in trials])

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"best params : {best_hps.values}")

run = train("tuned", **best_hps.values)
add_result(run / "weights/best.pt", "tuned", run)

Porownanie modeli
Heatmapa

In [ ]:
df = pd.read_csv(RESULTS).set_index("model")
change = ((df - df.loc["baseline"]) / df.loc["baseline"] * 100).round(1)
display(df.round(4))
display(change[["mAP50-95", "F1", "ms_cpu", "train_kWh"]])

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ["mAP50-95", "F1", "ms_cpu", "train_kWh"]):
    sns.barplot(x=df.index, y=df[col], ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(OUT / "comparison.png", dpi=150)
plt.show()

per_class = pd.read_csv(PER_CLASS).pivot_table(index="class", columns="model", values="mAP50-95")
plt.figure(figsize=(10, 5))
sns.heatmap(per_class[df.index], annot=True, fmt=".3f", cmap="Greens")
plt.title("mAP50-95 per klasa (test)")
plt.tight_layout()
plt.savefig(OUT / "per_class.png", dpi=150)
plt.show()